# Módulo 6 · Práctica: comprimir es predecir

Vas a convertir el modelo de bigramas del módulo 1 en un **compresor de texto**. Primero mides la información con entropía, entropía cruzada y KL. Después construyes un código de Huffman y un codificador aritmético exacto, y compruebas que el número de bits sale casi igual a −log₂ P(texto). Es la misma métrica (bits por byte) con la que se comparan los LLM.

**Cómo trabajar:** ejecuta en orden. Completa las celdas `# TU CÓDIGO` y ejecuta la `# Comprobación` que va detrás.

| Parte | Qué haces |
| --- | --- |
| 0 | Sorpresa en bits |
| 1 | Entropía |
| 2 | Entropía cruzada y KL |
| 3 | Texto, modelo de unigramas y código de Huffman |
| 4 | Modelo de bigramas y longitud ideal |
| 5 | Codificación aritmética exacta |
| 6 | Comparativa: bits por carácter y bits por byte |
| 7 | Información mutua entre letras vecinas |
| 8 | Retos |

## Parte 0 · Sorpresa en bits

La sorpresa de un suceso de probabilidad p es $h(p) = -\log_2 p$ bits.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

for p in [1, 0.5, 0.25, 1/8, 0.9, 0.1, 1/1024]:
    print(f"p = {p:<8.4g} sorpresa = {-np.log2(p):6.3f} bits")

## Parte 1 · Entropía

$$H(p) = -\sum_x p(x)\log_2 p(x)$$

Es la sorpresa media. Convención: $0 \log 0 = 0$.

**Ejercicio 1.** Implementa `entropia(p)`.

In [ ]:
def entropia(p):
    # TU CÓDIGO: H(p) = -sum p log2 p, ignorando los p = 0
    return None

In [ ]:
# Comprobación
assert entropia([0.5, 0.25, 0.25]) is not None, "Devuelve un número"
assert abs(entropia([0.5, 0.25, 0.25]) - 1.5) < 1e-9, "H(1/2, 1/4, 1/4) debe ser 1,5 bits"
assert abs(entropia(np.ones(6) / 6) - np.log2(6)) < 1e-9, "Un dado justo: log2 6 = 2,585 bits"
assert entropia([1, 0, 0]) == 0, "Una distribución segura tiene entropía 0 (cuidado con 0·log 0)"
print("Correcto")

## Parte 2 · Entropía cruzada y KL

$$H(p,q) = -\sum_x p(x)\log_2 q(x), \qquad D_{KL}(p\,\|\,q) = H(p,q) - H(p) = \sum_x p(x)\log_2\frac{p(x)}{q(x)}$$

**Ejercicio 2.** Implementa ambas.

In [ ]:
def entropia_cruzada(p, q):
    # TU CÓDIGO: H(p, q) = -sum p log2 q (solo donde p > 0)
    return None

def kl(p, q):
    # TU CÓDIGO: D(p||q) = H(p, q) - H(p)
    return None

In [ ]:
# Comprobación
p, q = [0.5, 0.25, 0.25], [0.8, 0.1, 0.1]
assert entropia_cruzada(p, q) is not None and kl(p, q) is not None, "Devuelve números"
assert abs(entropia_cruzada(p, q) - 1.8219) < 1e-4, "H(p, q) debe valer 1,822 bits"
assert abs(kl(p, q) - 0.3219) < 1e-4 and abs(kl(q, p) - 0.2781) < 1e-4, "KL no es simétrica: 0,322 frente a 0,278"
rng = np.random.default_rng(0)
for _ in range(1000):
    a, b = rng.dirichlet(np.ones(5)), rng.dirichlet(np.ones(5))
    assert kl(a, b) >= -1e-12, "Desigualdad de Gibbs: KL >= 0 siempre"
assert abs(kl(p, p)) < 1e-12, "KL(p||p) = 0"
print("Correcto: KL >= 0 en 1.000 pares aleatorios")

## Parte 3 · Texto, unigramas y Huffman

Usamos un texto corto en español escrito para el curso (no se puede descargar nada). Lo pasamos a minúsculas y trabajamos con caracteres, como en el módulo 1. El 75 % inicial sirve para **entrenar** (contar) y el 25 % final para **comprimir**.

In [ ]:
TEXTO = """En un pueblo de la meseta vivía una maestra que enseñaba a leer a los niños con un juego muy sencillo. Escribía en la pizarra el principio de una palabra y preguntaba qué letra venía después. Si escribía «casti», todos gritaban «llo» sin pensarlo. Si escribía «qu», nadie dudaba de que la siguiente letra sería una «e» o una «i». La maestra decía que adivinar bien era la mitad de saber leer.
Un día llegó al pueblo un telegrafista. Cobraba cada palabra que enviaba y por eso odiaba las letras inútiles. Le contó a la maestra que, si dos personas comparten las mismas expectativas, pueden ahorrarse lo que ya es obvio. Basta con enviar lo que sorprende. Lo previsible no lleva información, y lo que no lleva información no hace falta pagarlo.
La maestra pensó que aquello era lo mismo que hacían sus alumnos. Cuando un niño adivina la letra siguiente, esa letra no le enseña nada nuevo. Cuando falla, aprende algo. La sorpresa es la medida de lo que se aprende. El telegrafista sonrió y dijo que en su oficio la sorpresa se medía en bits.
Pensaron entonces en una máquina que hiciera lo mismo. La máquina leería millones de libros y aprendería a predecir la letra siguiente, o la palabra siguiente, con mucha precisión. Una máquina así sería también el mejor compresor posible, porque solo tendría que guardar sus errores. Y quizá, si predijera lo bastante bien, parecería entender lo que lee.
La maestra no vivió para ver esas máquinas, pero sus alumnos sí. Hoy escriben a un programa una pregunta y el programa responde letra a letra, eligiendo cada vez la continuación más probable. Algunos dicen que el programa solo adivina. Otros contestan que adivinar bien es la mitad de saber leer, y que la otra mitad todavía nadie la ha medido.
"""

texto = TEXTO.lower()
vocab = sorted(set(texto)); V = len(vocab)
stoi = {c: i for i, c in enumerate(vocab)}
corte = int(0.75 * len(texto))
train, test = texto[:corte], texto[corte:]
print(f"{V} símbolos · {len(train)} caracteres de entrenamiento · {len(test)} de prueba")
print("Un código de longitud fija necesitaría", int(np.ceil(np.log2(V))), "bits/carácter")

In [ ]:
from collections import Counter
cuentas = Counter(train)
p_uni = np.array([cuentas[c] for c in vocab], float) + 1      # Laplace: ningún símbolo con p = 0
p_uni /= p_uni.sum()
print(f"Entropía del modelo de unigramas: {entropia(p_uni):.3f} bits/carácter")
print("Más frecuentes:", sorted(vocab, key=lambda c: -p_uni[stoi[c]])[:6])

**Ejercicio 3.** Completa el algoritmo de Huffman: fusiona repetidamente los dos nodos menos probables. Cada nodo guarda un diccionario con los códigos parciales de sus símbolos.

In [ ]:
import heapq

def huffman(probs):
    """probs: dict símbolo -> probabilidad. Devuelve dict símbolo -> código binario."""
    heap = [(p, i, {s: ''}) for i, (s, p) in enumerate(probs.items())]
    heapq.heapify(heap)
    contador = len(heap)          # desempata nodos con igual probabilidad
    while len(heap) > 1:
        # TU CÓDIGO: saca los dos nodos menos probables del montículo,
        # antepón '0' a los códigos del primero y '1' a los del segundo,
        # y vuelve a meter el nodo fusionado (prob. suma, contador, dict).
        # Incrementa contador.
        break
    return heap[0][2]

In [ ]:
# Comprobación
codigo = huffman({c: p_uni[stoi[c]] for c in vocab})
assert len(codigo) == V, "Debe haber un código por símbolo"
cods = sorted(codigo.values())
assert all(not b.startswith(a) for a, b in zip(cods, cods[1:])), "Ningún código puede ser prefijo de otro"
L_media = sum(p_uni[stoi[c]] * len(codigo[c]) for c in vocab)
assert entropia(p_uni) <= L_media < entropia(p_uni) + 1, "Huffman cumple H <= L < H + 1"
bits_huff = sum(len(codigo[c]) for c in test)
print(f"Correcto. Longitud media {L_media:.3f} bits frente a H = {entropia(p_uni):.3f}")
print(f"Texto de prueba: {bits_huff} bits = {bits_huff / len(test):.3f} bits/carácter")
print("Código de ' ':", codigo[' '], "· código de 'z':", codigo['z'])

## Parte 4 · Bigramas y longitud ideal

Con el modelo de bigramas del módulo 1 (suavizado de Laplace, α = 1), la longitud ideal de un texto es $-\log_2 P(\text{texto}) = \sum_t -\log_2 P(x_t \mid x_{t-1})$ bits.

**Ejercicio 4.** Implementa `bits_ideales(P, s, ctx)`, donde `ctx` es el carácter anterior al primero de `s`.

In [ ]:
N = np.zeros((V, V), dtype=int)
for a, b in zip(train, train[1:]):
    N[stoi[a], stoi[b]] += 1
C = N + 1                                  # conteos suavizados (enteros)
P = C / C.sum(axis=1, keepdims=True)

In [ ]:
def bits_ideales(P, s, ctx):
    # TU CÓDIGO: suma -log2 P[ctx, c] para cada carácter c de s,
    # actualizando el contexto ctx = c tras cada paso
    return None

In [ ]:
# Comprobación
ideal_uni = -np.log2(p_uni[[stoi[c] for c in test]]).sum()
ideal_bi = bits_ideales(P, test, train[-1])
assert ideal_bi is not None, "Devuelve un número"
assert abs(bits_ideales(P, "a", "l") + np.log2(P[stoi['l'], stoi['a']])) < 1e-12, "Un solo carácter: -log2 P(a|l)"
assert 1500 < ideal_bi < 1650, "El texto de prueba debería costar unos 1.578 bits"
print(f"Correcto. Ideal unigramas: {ideal_uni:.1f} bits · ideal bigramas: {ideal_bi:.1f} bits")

## Parte 5 · Codificación aritmética exacta

Cada carácter estrecha el intervalo [low, low + ancho) en proporción a su probabilidad. Al final se envía el número binario más corto dentro del intervalo: ocupa como mucho ⌈−log₂ ancho⌉ bits. Usamos `Fraction` para que no haya errores de redondeo.

**Ejercicio 5.** Completa la actualización del intervalo en `codificar`: el símbolo j de la fila i ocupa el subintervalo que empieza en `cum[i, j] / tot[i]` y mide `C[i, j] / tot[i]`.

In [ ]:
from fractions import Fraction
import math

cum = np.concatenate([np.zeros((V, 1), dtype=int), np.cumsum(C, axis=1)], axis=1)
tot = cum[:, -1]   # cum[i, j] = conteos acumulados antes de j en la fila i

def codificar(s, ctx):
    low, ancho = Fraction(0), Fraction(1)
    for c in s:
        i, j = stoi[ctx], stoi[c]
        # TU CÓDIGO: actualiza low y ancho (dos líneas, con Fraction)
        pass
        ctx = c
    # el número binario más corto m / 2^k dentro de [low, low + ancho)
    k = 0
    while True:
        m = -((-low.numerator * 2**k) // low.denominator)   # techo de low * 2^k
        if Fraction(m, 2**k) < low + ancho:
            return format(m, 'b').zfill(k) if k > 0 else ''
        k += 1

def decodificar(bits, ctx, n):
    x = Fraction(int(bits, 2) if bits else 0, 2**len(bits))
    low, ancho, salida = Fraction(0), Fraction(1), []
    for _ in range(n):
        i = stoi[ctx]
        v = math.floor((x - low) / ancho * int(tot[i]))
        j = int(np.searchsorted(cum[i], v, side='right') - 1)
        low = low + ancho * Fraction(int(cum[i, j]), int(tot[i]))
        ancho = ancho * Fraction(int(C[i, j]), int(tot[i]))
        ctx = vocab[j]
        salida.append(ctx)
    return ''.join(salida)

t0 = time.time()
mensaje = codificar(test, train[-1])
bits_arit = len(mensaje)
print(f"{bits_arit} bits ({bits_arit / len(test):.3f} bits/carácter) en {time.time() - t0:.2f} s")


In [ ]:
# Comprobación
assert decodificar(mensaje, train[-1], len(test)) == test, "Al decodificar debe salir el texto original"
assert abs(bits_arit - ideal_bi) <= 2, "La longitud debe quedar a menos de 2 bits de -log2 P(texto)"
print(f"Correcto. Decodificación exacta. Real {bits_arit} bits frente a ideal {ideal_bi:.1f}")

## Parte 6 · Comparativa

Los LLM se comparan en **bits por byte** (bpb): bits totales divididos por los bytes UTF-8 del texto. No depende del tokenizador. Las tildes, la ñ y las comillas « » ocupan 2 bytes.

**Ejercicio 6.** Calcula `bpb` del codificador aritmético.

In [ ]:
# TU CÓDIGO: bits por byte del codificador aritmético sobre test (UTF-8)
bpb = None

In [ ]:
# Comprobación
import zlib
nbytes = len(test.encode('utf-8'))
assert bpb is not None and abs(bpb - bits_arit / nbytes) < 1e-12, "bits / bytes UTF-8"
filas = [("Longitud fija", np.ceil(np.log2(V)) * len(test)),
         ("Huffman unigramas", bits_huff),
         ("Ideal unigramas", ideal_uni),
         ("Ideal bigramas", ideal_bi),
         ("Aritmética bigramas", bits_arit),
         ("zlib nivel 9", 8 * len(zlib.compress(test.encode('utf-8'), 9)))]
print(f"{len(test)} caracteres, {nbytes} bytes\n")
for nombre, b in filas:
    print(f"{nombre:<22}{b:8.0f} bits  {b / len(test):5.3f} bits/car  {b / nbytes:5.3f} bpb")
plt.barh([f[0] for f in filas][::-1], [f[1] / len(test) for f in filas][::-1])
plt.xlabel("bits por carácter"); plt.title("Mejor predicción, menos bits"); plt.show()

## Parte 7 · Información mutua

$$I(X;Y) = \sum_{x,y} p(x,y)\log_2\frac{p(x,y)}{p(x)p(y)} = H(Y) - H(Y\mid X)$$

Mide cuántos bits sobre la letra siguiente te da conocer la actual.

**Ejercicio 7.** Calcula I entre letras consecutivas de `train` a partir de los conteos `N` (sin suavizar).

In [ ]:
# TU CÓDIGO: Pxy = N / N.sum(); marginales px, py; I = sum Pxy log2(Pxy / (px py))
I = None

In [ ]:
# Comprobación
assert I is not None and 1.0 < I < 1.5, "Debería salir en torno a 1,30 bits"
Hy = entropia(py)
print(f"Correcto. I = {I:.3f} bits. H(Y) = {Hy:.3f}, H(Y|X) = {Hy - I:.3f}")
print("Conocer la letra anterior ahorra", round(100 * I / Hy), "% de la incertidumbre")

## Parte 8 · Retos

1. **Ajusta α.** Repite la parte 4 con α ∈ {0,01; 0,1; 0,5; 1}. ¿Cuál comprime mejor el texto de prueba? (Con α = 0,1 deberías llegar a unos 3,35 bits/carácter.) Para la codificación aritmética necesitas conteos enteros: multiplica todo por 10.
2. **Trigramas.** Usa dos caracteres de contexto. ¿Mejora o empeora con tan pocos datos? Relaciónalo con el sobreajuste del módulo 1.
3. **Modelo adaptativo.** Actualiza los conteos *mientras* codificas; el decodificador hace lo mismo y no hace falta enviar el modelo.
4. **Huffman por pares.** Construye Huffman sobre pares de caracteres. ¿Se acerca más a la entropía? ¿Por qué la aritmética no necesita este truco?
5. **¿Es justo con zlib?** El modelo vio `train` y zlib no. ¿Qué «entrenamiento previo» lleva un LLM al comprimir?

---
### Soluciones de referencia
Inténtalo antes de mirar.

In [ ]:
# --- Ejercicio 1 ---
# def entropia(p):
#     p = np.asarray(p, dtype=float)
#     p = p[p > 0]
#     return float(-(p * np.log2(p)).sum())
# --- Ejercicio 2 ---
# def entropia_cruzada(p, q):
#     p, q = np.asarray(p, float), np.asarray(q, float)
#     m = p > 0
#     return float(-(p[m] * np.log2(q[m])).sum())
# 
# def kl(p, q):
#     return entropia_cruzada(p, q) - entropia(p)
# --- Ejercicio 3 ---
#         p0, _, a = heapq.heappop(heap)
#         p1, _, b = heapq.heappop(heap)
#         nuevo = {s: '0' + c for s, c in a.items()}
#         nuevo.update({s: '1' + c for s, c in b.items()})
#         heapq.heappush(heap, (p0 + p1, contador, nuevo))
#         contador += 1
# --- Ejercicio 4 ---
# def bits_ideales(P, s, ctx):
#     total = 0.0
#     for c in s:
#         total += -np.log2(P[stoi[ctx], stoi[c]])
#         ctx = c
#     return total
# --- Ejercicio 5 ---
#         low = low + ancho * Fraction(int(cum[i, j]), int(tot[i]))
#         ancho = ancho * Fraction(int(C[i, j]), int(tot[i]))
# --- Ejercicio 6 ---
# bpb = bits_arit / len(test.encode('utf-8'))
# --- Ejercicio 7 ---
# Pxy = N / N.sum()
# px, py = Pxy.sum(axis=1), Pxy.sum(axis=0)
# m = Pxy > 0
# I = float((Pxy[m] * np.log2(Pxy[m] / np.outer(px, py)[m])).sum())